In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    KFold,
    cross_val_score
)

from sklearn.ensemble import (
    RandomForestRegressor
)

from xgboost import XGBRegressor

from lightgbm import LGBMRegressor

from catboost import CatBoostRegressor

In [4]:
train_df = pd.read_csv(
    "../data/processed/train_after_feature_engineering.csv"
)

test_df = pd.read_csv(
    "../data/processed/test_after_feature_engineering.csv"
)

In [5]:
X = train_df.drop("SalePrice", axis=1)

y = np.log1p(train_df["SalePrice"])

In [6]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [7]:
def rmse_cv(model):

    rmse = np.sqrt(
        -cross_val_score(
            model,
            X,
            y,
            scoring="neg_mean_squared_error",
            cv=kf
        )
    )

    return rmse

In [8]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_scores = rmse_cv(rf_model)

print("Random Forest")

print("Mean RMSE:", rf_scores.mean())

print("Std RMSE:", rf_scores.std())

Random Forest
Mean RMSE: 0.14706525161492082
Std RMSE: 0.015663371699504186


In [9]:
lgbm_model = LGBMRegressor(
    learning_rate=0.01,
    n_estimators=3000,
    num_leaves=20,
    random_state=42
)

lgbm_scores = rmse_cv(lgbm_model)

print("LightGBM")

print("Mean RMSE:", lgbm_scores.mean())

print("Std RMSE:", lgbm_scores.std())

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003408 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1953
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 111
[LightGBM] [Info] Start training from score 12.030658
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000726 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1952
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 113
[LightGBM] [Info] Start training from score 12.016898
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of 

In [11]:
xgb_model = XGBRegressor(
    learning_rate=0.01,
    n_estimators=3000,
    max_depth=3,
    random_state=42
)

xgb_scores = rmse_cv(xgb_model)

print("XGBoost")

print("Mean RMSE:", xgb_scores.mean())

print("Std RMSE:", xgb_scores.std())

XGBoost
Mean RMSE: 0.14051636258254216
Std RMSE: 0.016254417898825445


In [12]:
cat_model = CatBoostRegressor(
    verbose=0,
    learning_rate=0.01,
    iterations=3000,
    random_seed=42
)

cat_scores = rmse_cv(cat_model)

print("CatBoost")

print("Mean RMSE:", cat_scores.mean())

print("Std RMSE:", cat_scores.std())

CatBoost
Mean RMSE: 0.13490807298914004
Std RMSE: 0.015228826978206959


In [13]:
all_results = pd.DataFrame({

    "Model": [
        "RandomForest",
        "LightGBM",
        "XGBoost",
        "CatBoost"
    ],

    "CV RMSE": [
        rf_scores.mean(),
        lgbm_scores.mean(),
        xgb_scores.mean(),
        cat_scores.mean()
    ],

    "CV Std": [
        rf_scores.std(),
        lgbm_scores.std(),
        xgb_scores.std(),
        cat_scores.std()
    ]
})

all_results = all_results.sort_values(
    by="CV RMSE",
    ascending=True
)

all_results

,Model,CV RMSE,CV Std
3,CatBoost,0.134908,0.015229
2,XGBoost,0.140516,0.016254
1,LightGBM,0.145437,0.017555
0,RandomForest,0.147065,0.015663


In [14]:
all_results.to_csv(
    "../model_reports/boosting_results.csv",
    index=False
)